In [20]:
%%writefile app.py
import pandas as pd
import streamlit as st

from utils import (
    FEATURES,
    SCENARIOS,
    difference_explanation,
    load_test_data,
    predict_ai,
    rule_based_decision,
    style_result,
)

# -----------------------------------
# 기본 설정
# -----------------------------------
st.set_page_config(
    page_title="Rule-based vs AI-based 의사결정 비교",
    page_icon="🏭",
    layout="wide"
)

# -----------------------------------
# 기본 멀티페이지 사이드바 숨기기
# (pages 폴더가 있어도 상단 app / CSV 비교 / 직접 입력 비교 안 보이게)
# -----------------------------------
st.markdown(
    """
    <style>
    [data-testid="stSidebarNav"] {
        display: none;
    }

    /* 사이드바 버튼 간격 조금 정리 */
    section[data-testid="stSidebar"] .stButton {
        width: 100%;
    }

    section[data-testid="stSidebar"] .stButton > button {
        width: 100%;
        border-radius: 10px;
        padding-top: 0.75rem;
        padding-bottom: 0.75rem;
        font-weight: 600;
    }
    </style>
    """,
    unsafe_allow_html=True
)

# -----------------------------------
# 세션 상태 초기화
# -----------------------------------
if "selected_page" not in st.session_state:
    st.session_state.selected_page = "intro"

# -----------------------------------
# 공통 스타일
# -----------------------------------
def highlight_unknown_row(row):
    """
    Rule-based 결과가 '판단못함'인 경우
    해당 행 전체를 노란색으로 강조
    """
    if row.get("rule_based_result", "") == "판단못함":
        return ["background-color: #fff3b0"] * len(row)
    return [""] * len(row)


def validate_uploaded_dataframe(df: pd.DataFrame):
    """
    업로드된 파일의 컬럼 구조가 FEATURES와 동일한지 검사
    """
    actual_cols = list(df.columns)
    expected_cols = FEATURES

    if actual_cols != expected_cols:
        return False, (
            "업로드한 파일의 컬럼 구조가 올바르지 않습니다.\n\n"
            f"- 현재 컬럼: {actual_cols}\n"
            f"- 필요한 컬럼: {expected_cols}"
        )
    return True, ""


def run_decision_on_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    입력 데이터프레임에 대해 Rule-based / AI-based 결과 계산
    """
    result_df = df.copy()

    rule_results = []
    rule_reasons = []
    ai_results = []

    for _, row in result_df.iterrows():
        row_dict = row.to_dict()

        rule_result, rule_reason = rule_based_decision(row_dict)
        ai_result, _ = predict_ai(row_dict)

        rule_results.append(rule_result)
        rule_reasons.append(rule_reason)
        ai_results.append(ai_result)

    result_df["rule_based_result"] = rule_results
    result_df["rule_reason"] = rule_reasons
    result_df["ai_based_result"] = ai_results
    result_df["same_or_diff"] = result_df.apply(
        lambda x: "같음" if x["rule_based_result"] == x["ai_based_result"] else "다름",
        axis=1
    )

    return result_df


# -----------------------------------
# 사이드바 페이지 버튼
# -----------------------------------
def render_sidebar_navigation():
    st.sidebar.title("메뉴")
    st.sidebar.markdown("### 페이지 선택")

    intro_type = "primary" if st.session_state.selected_page == "intro" else "secondary"
    test_type = "primary" if st.session_state.selected_page == "test" else "secondary"
    user_type = "primary" if st.session_state.selected_page == "user" else "secondary"
    mfg_type = "primary" if st.session_state.selected_page == "mfg_quality" else "secondary"

    if st.sidebar.button(
        "Rule-based .vs. AI-based 웹앱 소개",
        use_container_width=True,
        type=intro_type,
        key="btn_intro"
    ):
        st.session_state.selected_page = "intro"

    if st.sidebar.button(
        "Test 데이터로 비교해보기",
        use_container_width=True,
        type=test_type,
        key="btn_test"
    ):
        st.session_state.selected_page = "test"

    if st.sidebar.button(
        "사용자 입력으로 비교해보기",
        use_container_width=True,
        type=user_type,
        key="btn_user"
    ):
        st.session_state.selected_page = "user"

    if st.sidebar.button(
        "제조데이터 업로드 & 품질검사",
        use_container_width=True,
        type=mfg_type,
        key="btn_mfg_quality"
    ):
        st.session_state.selected_page = "mfg_quality"


# -----------------------------------
# 소개 페이지
# -----------------------------------
def show_intro_page():
    st.title("🏭 Rule-based .vs. AI-based 웹앱 소개")

    st.markdown("""
이 웹앱은 **Rule-based 의사결정**과 **AI-based 의사결정**의 차이를  
현업 담당자가 직접 체험할 수 있도록 만든 교육용 실습 도구입니다.

### 이 웹앱의 목적
- 사람이 만든 **규칙 기반 판단**이 어떻게 동작하는지 이해
- **AI가 학습 데이터를 기반으로 판단**하는 방식 이해
- 두 방식이 **언제 같고, 언제 달라지는지** 직접 비교
- 현업에서 왜 AI 기반 의사결정이 필요한지 직관적으로 이해

---

### 구성
#### 1) Test 데이터로 비교해보기
- 준비된 CSV 테스트 데이터를 불러옵니다.
- 각 행에 대해 **Rule-based 결과**와 **AI-based 결과**를 동시에 비교합니다.
- 특히 Rule-based에서 **'판단못함'** 인 케이스를 쉽게 확인할 수 있습니다.

#### 2) 사용자 입력으로 비교해보기
- 사용자가 슬라이더로 직접 입력값을 조절합니다.
- Rule-based와 AI-based의 결과를 즉시 비교할 수 있습니다.
- 추천 시나리오를 불러와 차이를 설명할 수 있습니다.

#### 3) 제조데이터 품질검사
- CSV 파일을 업로드합니다.
- 업로드한 제조데이터를 표로 확인합니다.
- 버튼을 눌러 Rule-based와 AI-based 품질검사를 동시에 수행합니다.

---

### 실습에서 전달하고 싶은 핵심 메시지
#### Rule-based의 장점
- 이해하기 쉽습니다.
- 기준이 명확합니다.
- 바로 적용할 수 있습니다.

#### Rule-based의 한계
- 규칙에 없는 상황은 판단하지 못합니다.
- 복합적인 패턴 반영이 어렵습니다.
- 새로운 변수(예: 습도)가 들어오면 다시 규칙을 수정해야 합니다.

#### AI-based의 장점
- 여러 변수의 조합 패턴을 반영할 수 있습니다.
- 사람이 명시적으로 규칙을 쓰지 않은 경우도 판정할 수 있습니다.
- 복잡한 중간 영역을 다룰 수 있습니다.

#### 주의할 점
- AI가 항상 더 낫다는 뜻은 아닙니다.
- 실제 현업에서는 **Rule-based + AI-based 보완 구조**가 현실적입니다.
""")

    st.info("왼쪽 사이드바에서 페이지를 선택해 주세요.")


# -----------------------------------
# Test 데이터 비교 페이지
# -----------------------------------
def show_test_compare_page():
    st.title("📄 Test 데이터로 비교해보기")
    st.caption("CSV 테스트 데이터를 불러와 Rule-based와 AI-based 결과를 비교합니다.")

    df = load_test_data().copy()

    rule_results = []
    rule_reasons = []
    ai_results = []

    for _, row in df.iterrows():
        row_dict = row.to_dict()

        rule_result, rule_reason = rule_based_decision(row_dict)
        ai_result, _ = predict_ai(row_dict)

        rule_results.append(rule_result)
        rule_reasons.append(rule_reason)
        ai_results.append(ai_result)

    df["rule_based_result"] = rule_results
    df["rule_reason"] = rule_reasons
    df["ai_based_result"] = ai_results
    df["same_or_diff"] = df.apply(
        lambda x: "같음" if x["rule_based_result"] == x["ai_based_result"] else "다름",
        axis=1
    )

    st.markdown("## 전체 비교 결과")

    styled_df = df[
        FEATURES + ["rule_based_result", "rule_reason", "ai_based_result", "same_or_diff"]
    ].style.apply(highlight_unknown_row, axis=1)

    st.dataframe(
        styled_df,
        use_container_width=True,
        hide_index=False
    )

    st.warning("노란색으로 표시된 행은 Rule-based에서 '판단못함'으로 처리된 케이스입니다.")

    st.markdown("---")

    left, right = st.columns(2)

    with left:
        st.subheader("Rule-based 결과")
        styled_rule_df = df[
            FEATURES + ["rule_based_result", "rule_reason"]
        ].style.apply(highlight_unknown_row, axis=1)

        st.dataframe(
            styled_rule_df,
            use_container_width=True,
            hide_index=False
        )

    with right:
        st.subheader("AI-based 결과")
        st.dataframe(
            df[FEATURES + ["ai_based_result"]],
            use_container_width=True,
            hide_index=False
        )

    st.markdown("---")
    st.subheader("Rule-based가 판단못함인 케이스만 보기")

    unknown_df = df[df["rule_based_result"] == "판단못함"].copy()

    if unknown_df.empty:
        st.success("현재 테스트 데이터에는 '판단못함' 케이스가 없습니다.")
    else:
        styled_unknown_df = unknown_df[
            FEATURES + ["rule_based_result", "rule_reason", "ai_based_result"]
        ].style.apply(highlight_unknown_row, axis=1)

        st.dataframe(
            styled_unknown_df,
            use_container_width=True,
            hide_index=False
        )

        for idx, row in unknown_df.iterrows():
            with st.expander(f"샘플 {idx + 1} 상세 해설"):
                row_dict = row[FEATURES].to_dict()

                col1, col2 = st.columns(2)

                with col1:
                    st.markdown("### Rule-based")
                    st.metric("판정", style_result(row["rule_based_result"]))
                    st.write(row["rule_reason"])

                with col2:
                    st.markdown("### AI-based")
                    st.metric("판정", style_result(row["ai_based_result"]))
                    _, probs = predict_ai(row_dict)
                    if probs:
                        prob_df = pd.DataFrame({
                            "class": list(probs.keys()),
                            "probability": list(probs.values())
                        })
                        st.bar_chart(prob_df.set_index("class"))

                st.markdown("### 해설")
                messages = difference_explanation(
                    row_dict,
                    row["rule_based_result"],
                    row["ai_based_result"]
                )
                for msg in messages:
                    st.write("- " + msg)

    st.markdown("---")
    st.subheader("샘플 하나씩 상세 확인")

    selected_idx = st.selectbox(
        "확인할 샘플 선택",
        options=list(range(len(df))),
        format_func=lambda x: f"샘플 {x + 1}"
    )

    selected = df.iloc[selected_idx]
    row_dict = selected[FEATURES].to_dict()

    col1, col2 = st.columns(2)

    with col1:
        st.markdown("### Rule-based")
        st.metric("판정", style_result(selected["rule_based_result"]))
        st.write(selected["rule_reason"])

    with col2:
        st.markdown("### AI-based")
        st.metric("판정", style_result(selected["ai_based_result"]))
        _, probs = predict_ai(row_dict)
        if probs:
            prob_df = pd.DataFrame({
                "class": list(probs.keys()),
                "probability": list(probs.values())
            })
            st.bar_chart(prob_df.set_index("class"))

    st.markdown("### 자동 해설")
    for msg in difference_explanation(
        row_dict,
        selected["rule_based_result"],
        selected["ai_based_result"]
    ):
        st.write("- " + msg)


# -----------------------------------
# 사용자 입력 비교 페이지
# -----------------------------------
def show_user_input_page():
    st.title("🎛️ 사용자 입력으로 비교해보기")
    st.caption("슬라이더로 값을 조정하며 Rule-based와 AI-based 결과를 즉시 비교합니다.")

    # -------------------------------
    # 슬라이더 상태 초기화
    # -------------------------------
    default_manual = {
        "temperature": 70.0,
        "pressure": 50.0,
        "vibration": 5.0,
        "process_time": 60.0,
        "humidity": 50.0,
    }

    if "user_input_initialized" not in st.session_state:
        st.session_state.user_input_initialized = True
        st.session_state.temperature = default_manual["temperature"]
        st.session_state.pressure = default_manual["pressure"]
        st.session_state.vibration = default_manual["vibration"]
        st.session_state.process_time = default_manual["process_time"]
        st.session_state.humidity = default_manual["humidity"]

    def apply_scenario_values(values: dict):
        st.session_state.temperature = float(values["temperature"])
        st.session_state.pressure = float(values["pressure"])
        st.session_state.vibration = float(values["vibration"])
        st.session_state.process_time = float(values["process_time"])
        st.session_state.humidity = float(values["humidity"])

    scenario_name = st.selectbox(
        "추천 예시 불러오기",
        ["직접 입력"] + list(SCENARIOS.keys()),
        key="scenario_select"
    )

    # 추천 예시 적용 버튼
    col_btn1, col_btn2 = st.columns([1, 3])
    with col_btn1:
        if st.button("예시 적용", use_container_width=True):
            if scenario_name == "직접 입력":
                apply_scenario_values(default_manual)
            else:
                apply_scenario_values(SCENARIOS[scenario_name])

    with col_btn2:
        st.caption("추천 예시를 고른 뒤 '예시 적용' 버튼을 누르면 슬라이더 값이 해당 예시로 바뀝니다.")

    # -------------------------------
    # 입력 슬라이더
    # -------------------------------
    input_col1, input_col2 = st.columns(2)

    with input_col1:
        temperature = st.slider(
            "온도", 55.0, 90.0,
            key="temperature",
            step=0.1
        )
        pressure = st.slider(
            "압력", 35.0, 65.0,
            key="pressure",
            step=0.1
        )
        vibration = st.slider(
            "진동", 2.0, 9.5,
            key="vibration",
            step=0.1
        )

    with input_col2:
        process_time = st.slider(
            "가공시간", 40.0, 85.0,
            key="process_time",
            step=0.1
        )
        humidity = st.slider(
            "습도", 25.0, 80.0,
            key="humidity",
            step=0.1
        )

    row_dict = {
        "temperature": float(temperature),
        "pressure": float(pressure),
        "vibration": float(vibration),
        "process_time": float(process_time),
        "humidity": float(humidity),
    }

    st.subheader("현재 입력값")
    input_df = pd.DataFrame([row_dict])[FEATURES]
    st.dataframe(input_df, use_container_width=True, hide_index=False)

    # -------------------------------
    # Rule / AI 판정
    # -------------------------------
    rule_result, rule_reason = rule_based_decision(row_dict)
    ai_result, probs = predict_ai(row_dict)

    col1, col2 = st.columns(2)

    with col1:
        st.markdown("### Rule-based 결과")
        st.metric("판정", style_result(rule_result))
        st.write(rule_reason)

        st.markdown("#### Rule-based 특징")
        st.write("- 사람이 정의한 규칙만 적용합니다.")
        st.write("- 규칙에 없는 조합은 '판단못함'입니다.")
        st.write("- 현재 데모에서는 습도 조건을 보지 않습니다.")

    with col2:
        st.markdown("### AI-based 결과")
        st.metric("판정", style_result(ai_result))
        st.write("학습된 패턴을 바탕으로 현재 입력의 품질 상태를 예측합니다.")

        if probs:
            prob_df = pd.DataFrame({
                "class": list(probs.keys()),
                "probability": list(probs.values())
            })
            st.bar_chart(prob_df.set_index("class"))

    st.markdown("---")
    st.subheader("자동 해설")
    for msg in difference_explanation(row_dict, rule_result, ai_result):
        st.write("- " + msg)

    st.markdown("---")
    st.subheader("강의용 설명 포인트")

    if rule_result == "판단못함":
        st.info(
            "이 케이스는 Rule-based가 정의한 조건 바깥에 있으므로 판단을 내리지 못합니다. "
            "AI는 과거 데이터 패턴을 기반으로 판정을 시도합니다."
        )
    elif rule_result == ai_result:
        st.success(
            "이 케이스는 Rule-based와 AI-based가 같은 결론을 내렸습니다. "
            "명확한 상황에서는 Rule-based도 충분히 효과적입니다."
        )
    else:
        st.warning(
            "두 방식의 결과가 다릅니다. "
            "규칙의 단순성과 AI의 패턴 학습 차이를 설명하기 좋은 사례입니다."
        )


# -----------------------------------
# 제조데이터 품질검사 페이지
# -----------------------------------
def show_manufacturing_quality_page():
    st.title("🧪 제조데이터 업로드 & 품질검사")
    st.caption("CSV 파일을 업로드하고 Rule-based와 AI-based 품질검사 결과를 확인합니다.")

    st.markdown("### 1. 제조데이터 업로드")
    st.write("아래 CSV 파일을 업로드해 주세요.")

    uploaded_file = st.file_uploader(
        "CSV 파일 업로드",
        type=["csv"],
        key="manufacturing_quality_upload"
    )

    if uploaded_file is None:
        st.info("품질검사를 진행하려면 CSV 파일을 업로드해 주세요.")
        st.markdown("#### 필요한 컬럼 구조")
        st.code(", ".join(FEATURES))
        return

    try:
        upload_df = pd.read_csv(uploaded_file)
    except Exception as e:
        st.error(f"파일을 읽는 중 오류가 발생했습니다: {e}")
        return

    is_valid, error_msg = validate_uploaded_dataframe(upload_df)
    if not is_valid:
        st.error(error_msg)
        st.markdown("#### 필요한 컬럼 구조")
        st.code(", ".join(FEATURES))
        return

    st.markdown("### 2. 업로드한 파일 내용")
    st.dataframe(upload_df, use_container_width=True, hide_index=False)

    run_button = st.button(
        "품질검사 실시",
        type="primary",
        use_container_width=False,
        key="run_manufacturing_quality_check"
    )

    if run_button:
        result_df = run_decision_on_dataframe(upload_df)

        st.markdown("---")
        st.markdown("## 3. 품질검사 결과")

        col1, col2 = st.columns(2)

        with col1:
            st.subheader("Rule-based 결과")
            rule_df = result_df[FEATURES + ["rule_based_result", "rule_reason"]].copy()
            styled_rule_df = rule_df.style.apply(highlight_unknown_row, axis=1)
            st.dataframe(
                styled_rule_df,
                use_container_width=True,
                hide_index=False
            )

        with col2:
            st.subheader("AI-based 결과")
            ai_df = result_df[FEATURES + ["ai_based_result"]].copy()
            st.dataframe(
                ai_df,
                use_container_width=True,
                hide_index=False
            )

        unknown_count = (result_df["rule_based_result"] == "판단못함").sum()
        diff_count = (result_df["same_or_diff"] == "다름").sum()

        st.markdown("---")
        summary_col1, summary_col2, summary_col3 = st.columns(3)

        with summary_col1:
            st.metric("전체 데이터 수", len(result_df))
        with summary_col2:
            st.metric("Rule-based 판단못함", int(unknown_count))
        with summary_col3:
            st.metric("두 방식 결과 다름", int(diff_count))

        if unknown_count > 0:
            st.warning("노란색으로 표시된 행은 Rule-based에서 '판단못함'으로 처리된 케이스입니다.")


# -----------------------------------
# 실행
# -----------------------------------
render_sidebar_navigation()

if st.session_state.selected_page == "intro":
    show_intro_page()
elif st.session_state.selected_page == "test":
    show_test_compare_page()
elif st.session_state.selected_page == "user":
    show_user_input_page()
elif st.session_state.selected_page == "mfg_quality":
    show_manufacturing_quality_page()
else:
    show_intro_page()

Overwriting app.py


In [18]:
%%writefile utils.py

from pathlib import Path
import joblib
import pandas as pd
import streamlit as st

BASE_DIR = Path(__file__).resolve().parent
MODEL_PATH = BASE_DIR / "quality_model.pkl"
TEST_PATH = BASE_DIR / "test_quality_data.csv"

FEATURES = ["temperature", "pressure", "vibration", "process_time", "humidity"]
LABEL_ORDER = ["정상", "주의", "불량"]

SCENARIOS = {
    "습도 영향 케이스": {"temperature": 70.0, "pressure": 53.0, "vibration": 6.0, "process_time": 72.0, "humidity": 74.0},
    "복합 패턴 케이스": {"temperature": 78.0, "pressure": 54.0, "vibration": 6.9, "process_time": 69.0, "humidity": 72.0},
    "임계값 살짝 벗어난 케이스": {"temperature": 82.0, "pressure": 50.0, "vibration": 6.7, "process_time": 75.0, "humidity": 68.0},
    "명확한 정상 케이스": {"temperature": 60.0, "pressure": 40.0, "vibration": 3.2, "process_time": 50.0, "humidity": 45.0},
    "명확한 불량 케이스": {"temperature": 88.0, "pressure": 62.0, "vibration": 8.5, "process_time": 80.0, "humidity": 76.0},
}

def rule_based_decision(row):
    """
    Rule-based 품질 판정
    - 불량: temperature >= 82 and pressure >= 58
    - 주의: vibration >= 8.0
    - 정상: temperature <= 65 and pressure <= 45 and vibration <= 4.0 and process_time <= 55
    - 그 외: 판단못함
    """
    try:
        t = float(row.get("temperature"))
        p = float(row.get("pressure"))
        v = float(row.get("vibration"))
        pt = float(row.get("process_time"))
    except (TypeError, ValueError, AttributeError):
        return "판단못함", "입력값 형식이 올바르지 않아 Rule-based 판정을 수행할 수 없음"

    # 1) 불량
    if t >= 82 and p >= 58:
        return "불량", "온도 >= 82 이고 압력 >= 58 이므로 불량"

    # 2) 주의
    if v >= 8.0:
        return "주의", "진동 >= 8.0 이므로 주의"

    # 3) 정상
    if t <= 65 and p <= 45 and v <= 4.0 and pt <= 55:
        return "정상", "정상 기준(온도/압력/진동/가공시간)을 모두 만족"

    # 4) 규칙 외 케이스
    return "판단못함", "현재 입력은 정의된 규칙 범위에 없음"

@st.cache_resource
def load_model():
    return joblib.load(MODEL_PATH)

@st.cache_data
def load_test_data():
    return pd.read_csv(TEST_PATH)

def predict_ai(row_dict):
    model = load_model()
    X = pd.DataFrame([row_dict])[FEATURES]
    pred = model.predict(X)[0]

    probs = None
    if hasattr(model, "predict_proba"):
        raw = model.predict_proba(X)[0]
        classes = list(model.classes_)
        probs = {cls: float(raw[classes.index(cls)]) for cls in classes}
    return pred, probs

def difference_explanation(row_dict, rule_result, ai_result):
    messages = []

    if rule_result == "판단못함":
        messages.append("Rule-based는 현재 입력 조합에 해당하는 규칙이 없어 판단을 멈췄습니다.")
    if row_dict["humidity"] >= 70 or row_dict["humidity"] <= 30:
        messages.append("현재 케이스는 습도 영향이 큽니다. 하지만 Rule-based 규칙에는 습도 조건이 없습니다.")
    if 74 <= row_dict["temperature"] <= 82 and row_dict["vibration"] >= 6.2:
        messages.append("온도와 진동이 동시에 높아지는 복합 패턴입니다.")
    if row_dict["pressure"] >= 54 and row_dict["process_time"] >= 70:
        messages.append("압력과 가공시간이 함께 높아지는 패턴은 품질 리스크를 높일 수 있습니다.")

    if rule_result == ai_result:
        messages.append("이 케이스는 두 방식이 같은 결론을 내렸습니다. 명확한 영역에서는 Rule-based도 충분히 유효합니다.")
    else:
        messages.append(f"두 방식의 결과가 다릅니다. AI는 학습 데이터의 패턴을 바탕으로 '{ai_result}' 판정을 시도했습니다.")

    return messages

def style_result(result):
    if result == "정상":
        return "✅ 정상"
    elif result == "주의":
        return "🟡 주의"
    elif result == "불량":
        return "🔴 불량"
    return "⚪ 판단못함"

Overwriting utils.py


In [ ]:
!git add .
!git commit -am "파일 번호 이름만 바꿈"
!git push -u origin main


In [28]:
!git log -n 5

commit 8d2fcb3c6a12d12bb372111e4e66c14df34898e9 (HEAD -> main, origin/main)
Author: serendipi27 <serendipi27@gmail.com>
Date:   Sat Mar 28 11:52:29 2026 +0900

    파일추가 - 사용자입력으로비교해보기_업로드데이터2.csv

commit 6a7a34f10980390c03183a9880bad5c1a5e22f3b
Author: serendipi27 <serendipi27@gmail.com>
Date:   Sat Mar 28 11:35:30 2026 +0900

    revised func show_user_input_page() on app.py

commit 8f9d1e116deb939c02fe270f66469e707ff26a0a
Author: serendipi27 <serendipi27@gmail.com>
Date:   Sat Mar 28 11:21:20 2026 +0900

    revised utils.py

commit 8672213882dfce6d6f628e3fb7bd01593aa0a2a8
Author: serendipi27 <serendipi27@gmail.com>
Date:   Sat Mar 28 11:08:11 2026 +0900

    사소한 몇가지 수정 app.py

commit 45cc82a93bda29ddc7ae0c78851909cda7caefa9
Author: serendipi27 <serendipi27@gmail.com>
Date:   Sat Mar 28 10:59:20 2026 +0900

    제조데이터 업로드 & 품질검사 페이지 추가 app.py


# Rollback 예시 - reset --hard
- 그냥 잘못된 작업이여서 아예 git history에서 삭제

In [24]:
!git reset --hard HEAD~1

HEAD is now at 6a7a34f revised func show_user_input_page() on app.py


In [29]:
!git reset --hard 8672213882dfce6d6f628e3fb7bd01593aa0a2a8b

fatal: ambiguous argument '8672213882dfce6d6f628e3fb7bd01593aa0a2a8b': unknown revision or path not in the working tree.
Use '--' to separate paths from revisions, like this:
'git <command> [<revision>...] -- [<file>...]'


In [27]:
!git push origin main --force

Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 8 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 8.10 KiB | 8.10 MiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To github.com:serendipi27/aiglue_rulebased_ai_demo.git
 + a3ad6da...8d2fcb3 main -> main (forced update)
